In [38]:
# Setup
import os, sys

# Get the directory where this notebook is located
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('main.ipynb'))
# ROOT is the parent of notebooks/ folder
ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

DATA = os.path.join(ROOT, 'data', 'entradaProj2.25TAG.txt')
FIGS = os.path.join(ROOT, 'notebooks', 'figs')
RESULTS = os.path.join(ROOT, 'results')

In [51]:
# Import libs (with reload to pick up code changes)
import importlib
import pandas as pd
import src.parser
import src.utils
import src.allocator
import src.visualizer
importlib.reload(src.parser)
importlib.reload(src.utils)
importlib.reload(src.allocator)
importlib.reload(src.visualizer)
from src.parser import load_input
from src.utils import build_project_prefs, rank_in_project, rank_in_student, is_stable, student_satisfaction, project_satisfaction, ensure_non_empty_projects
from src.allocator import run_gale_shapley
from src.visualizer import visualize_iteration

In [52]:
# Executar Gale-Shapley com 10 iterações fixas para visualização
projects, students = load_input(DATA)
matching, logs = run_gale_shapley(students, projects, max_iter=1000, fixed_iterations=10)

print(f"Visualização das 10 iterações do algoritmo Gale-Shapley:")
print(f"Total de alunos: {len(students)}")
print(f"Total de projetos: {len(projects)}")
print(f"Total de vagas: {sum(p.vacancies for p in projects.values())}")
print()

# Mostrar detalhes de cada iteração e gerar visualizações
for i in range(1, 11):
    state = logs.get(i, {'proposals': [], 'accepted': [], 'rejected': []})
    print(f"Iteração {i}: {len(state['proposals'])} propostas, {len(state['accepted'])} aceitos temporariamente, {len(state['rejected'])} rejeitados")
    out = os.path.join(FIGS, f'iter_{i:02d}.png')
    visualize_iteration(students, projects, state, i, out)

print(f"\nApós 10 iterações: {sum(1 for v in matching.values() if v is not None)} alunos emparelhados")

Visualização das 10 iterações do algoritmo Gale-Shapley:
Total de alunos: 200
Total de projetos: 50
Total de vagas: 80

Iteração 1: 200 propostas, 43 aceitos temporariamente, 157 rejeitados
Iteração 2: 157 propostas, 52 aceitos temporariamente, 148 rejeitados
Iteração 2: 157 propostas, 52 aceitos temporariamente, 148 rejeitados
Iteração 3: 144 propostas, 58 aceitos temporariamente, 138 rejeitados
Iteração 3: 144 propostas, 58 aceitos temporariamente, 138 rejeitados
Iteração 4: 9 propostas, 58 aceitos temporariamente, 9 rejeitados
Iteração 4: 9 propostas, 58 aceitos temporariamente, 9 rejeitados
Iteração 5: 3 propostas, 58 aceitos temporariamente, 3 rejeitados
Iteração 5: 3 propostas, 58 aceitos temporariamente, 3 rejeitados
Iteração 6: 0 propostas, 58 aceitos temporariamente, 0 rejeitados
Iteração 6: 0 propostas, 58 aceitos temporariamente, 0 rejeitados
Iteração 7: 0 propostas, 58 aceitos temporariamente, 0 rejeitados
Iteração 7: 0 propostas, 58 aceitos temporariamente, 0 rejeitados
It

In [48]:
# Pós-processamento: Garantir que cada projeto tenha pelo menos 1 aluno (requisito da especificação)
print("=== Antes do pós-processamento ===")
print(f"  Alunos emparelhados: {sum(1 for v in matching.values() if v is not None)}")
empty_before = sum(1 for p in projects.values() if not p.current_alloc)
print(f"  Projetos vazios: {empty_before}")

# Análise: Por que alguns projetos estão vazios?
print("\n=== Análise de projetos vazios ===")
for proj in projects.values():
    if not proj.current_alloc:
        eligible = sum(1 for s in students.values() if s.score >= proj.min_req)
        unmatched_eligible = sum(1 for s in students.values() if s.score >= proj.min_req and matching.get(s.id) is None)
        print(f"  {proj.code}: min_req={proj.min_req}, elegíveis={eligible}, não-emparelhados elegíveis={unmatched_eligible}")

still_empty = ensure_non_empty_projects(projects, students, matching)

print(f"\n=== Após pós-processamento ===")
print(f"  Alunos emparelhados: {sum(1 for v in matching.values() if v is not None)}")
print(f"  Projetos ainda vazios: {still_empty}")
if still_empty > 0:
    print(f"  NOTA: {still_empty} projetos permanecem vazios pois não há alunos elegíveis suficientes (min_req > notas disponíveis)")

# Gerar matriz de emparelhamento conforme especificação
rows = []
for sid, proj_code in sorted(matching.items()):
    if proj_code is None: 
        continue
    proj = projects[proj_code]
    stu = students[sid]
    
    # Rank do aluno na lista de preferência do projeto
    proj_rank = rank_in_project(proj, sid)
    total_in_proj_list = len(proj.pref_list)
    
    # Rank do projeto na lista de preferência do aluno
    stu_rank = rank_in_student(stu, proj_code)
    
    rows.append({
        'Aluno': f'A{sid}',
        'Projeto Emparelhado': proj_code,
        'Rank do Aluno (Lista do Projeto)': f'{proj_rank}º (de {total_in_proj_list} elegíveis)' if proj_rank < 10**9 else 'N/A',
        'Rank do Projeto (Lista do Aluno)': f'{stu_rank}ª escolha' if stu_rank <= 3 else 'Não listado',
    })

df = pd.DataFrame(rows)
os.makedirs(RESULTS, exist_ok=True)
df.to_csv(os.path.join(RESULTS, 'final_matching.csv'), index=False)

print("\n=== Matriz de Emparelhamento Final ===")
print(df.to_string(index=False))

# Métricas
print("\n=== Métricas ===")
print('Estável:', is_stable(matching, projects, students))
sat = student_satisfaction(matching, students)
print(f"Satisfação dos alunos:")
print(f"  1ª escolha: {sat['choice1_pct']:.1f}%")
print(f"  2ª escolha: {sat['choice2_pct']:.1f}%")
print(f"  3ª escolha: {sat['choice3_pct']:.1f}%")
print(f"  Não emparelhado/outro: {sat['other_pct']:.1f}%")
print(f"Satisfação dos projetos (rank médio dos alunos alocados): {project_satisfaction(projects):.2f}")

=== Antes do pós-processamento ===
  Alunos emparelhados: 58
  Projetos vazios: 11

=== Análise de projetos vazios ===
  P13: min_req=5, elegíveis=30, não-emparelhados elegíveis=0
  P19: min_req=5, elegíveis=30, não-emparelhados elegíveis=0
  P23: min_req=5, elegíveis=30, não-emparelhados elegíveis=0
  P31: min_req=5, elegíveis=30, não-emparelhados elegíveis=0
  P32: min_req=5, elegíveis=30, não-emparelhados elegíveis=0
  P33: min_req=5, elegíveis=30, não-emparelhados elegíveis=0
  P42: min_req=5, elegíveis=30, não-emparelhados elegíveis=0
  P44: min_req=5, elegíveis=30, não-emparelhados elegíveis=0
  P46: min_req=5, elegíveis=30, não-emparelhados elegíveis=0
  P48: min_req=5, elegíveis=30, não-emparelhados elegíveis=0
  P50: min_req=5, elegíveis=30, não-emparelhados elegíveis=0

=== Após pós-processamento ===
  Alunos emparelhados: 58
  Projetos ainda vazios: 1
  NOTA: 1 projetos permanecem vazios pois não há alunos elegíveis suficientes (min_req > notas disponíveis)

=== Matriz de Em

In [50]:
# Resumo dos resultados
print("=== RESUMO FINAL ===")
print(f"Total de alunos: {len(students)}")
print(f"Total de projetos: {len(projects)}")
print(f"Alunos emparelhados: {sum(1 for v in matching.values() if v is not None)}")
print(f"Projetos com pelo menos 1 aluno: {sum(1 for p in projects.values() if p.current_alloc)}")
empty_count = sum(1 for p in projects.values() if not p.current_alloc)
print(f"Projetos vazios: {empty_count}")

if empty_count > 0:
    print("\nProjetos que permaneceram vazios:")
    for p in projects.values():
        if not p.current_alloc:
            eligible = sum(1 for s in students.values() if s.score >= p.min_req)
            print(f"  {p.code}: min_req={p.min_req}, alunos elegíveis no total={eligible}")
    print("\nNOTA: Impossível preencher todos os projetos - há mais vagas exigindo nota 5 do que alunos com nota 5.")
    score5_students = sum(1 for s in students.values() if s.score == 5)
    score5_vacancies = sum(p.vacancies for p in projects.values() if p.min_req == 5)
    print(f"  Alunos com nota 5: {score5_students}")
    print(f"  Vagas que exigem nota 5: {score5_vacancies}")

# Mostrar os primeiros 10 emparelhamentos como exemplo
print("\n=== Primeiros 15 emparelhamentos (exemplo conforme especificação) ===")
print(df.head(15).to_string(index=False))

=== RESUMO FINAL ===
Total de alunos: 200
Total de projetos: 50
Alunos emparelhados: 58
Projetos com pelo menos 1 aluno: 49
Projetos vazios: 1

Projetos que permaneceram vazios:
  P50: min_req=5, alunos elegíveis no total=30

NOTA: Impossível preencher todos os projetos - há mais vagas exigindo nota 5 do que alunos com nota 5.
  Alunos com nota 5: 30
  Vagas que exigem nota 5: 35

=== Primeiros 15 emparelhamentos (exemplo conforme especificação) ===
Aluno Projeto Emparelhado Rank do Aluno (Lista do Projeto) Rank do Projeto (Lista do Aluno)
   A1                 P32                              N/A                      Não listado
   A2                  P1              2º (de 5 elegíveis)                       1ª escolha
   A6                  P9              1º (de 8 elegíveis)                       2ª escolha
   A8                  P6              1º (de 1 elegíveis)                       1ª escolha
  A10                 P43              1º (de 9 elegíveis)                       2ª es

## Geração do Mini-Relatório
Use nbconvert para exportar PDF:
````
jupyter nbconvert --to pdf notebooks/main.ipynb --output mini-relatorio.pdf
````